# Experimento en Colab L4: Edición `always_on` que Preserva Redacción Sana (`healthy_fisher` + Neuronas Esparsas + Capas Tardías)

Evalúa sobre **`LiquidAI/LFM3-1.7B`** (100% prompts benignos: `OR-Bench-Hard` + controles sanos, cero texto tóxico) si trazar la sensibilidad empírica de Fisher sobre respuestas sanas completas (`healthy_cost="fisher"`, `grad_mode="compressed"`) junto con edición esparsa de pocas neuronas (`16–128`) y capas tardías reduce el drift de `always_on` respecto al baseline (`healthy_kl = 0.00925`, `control_exact_match = 3.3%`, `19.9M` pesos) hacia `<= 0.001–0.002`, manteniendo alta recuperación de sobre-rechazo en `OR-Bench` Half-B.

Guarda el `.moipatch` esparso y el reporte completo en `MyDrive/wsi_runs/FINALES/prose_preserving_always_on_lfm3_l4.json`.

In [ ]:
MODEL = "LiquidAI/LFM3-1.7B"
RUN_NAME = "prose_preserving_always_on_lfm3_l4"
REPO = "/content/wsi"
BASE = "/content/base"
BASELINE_PATCH_DRIVE = "/content/drive/MyDrive/wsi_runs/surgical_refusal_lfm3/lfm3_safety_routed_refusal.moipatch"
PATCH_OUT_DRIVE = "/content/drive/MyDrive/wsi_runs/surgical_refusal_lfm3/lfm3_prose_preserving_sparse.moipatch"
N_OR = 160
N_CTRL = 60
N_TRACES = 8
MAX_NEW_TOK = 128
BATCH_SIZE = 16

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q torch transformers accelerate safetensors pandas pyarrow huggingface_hub
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
FINALES = "/content/drive/MyDrive/wsi_runs/FINALES"
RUN_DIR = "/content/drive/MyDrive/wsi_runs/surgical_refusal_lfm3"
os.makedirs(FINALES, exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True)

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/krexito/weight-space-index.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/main"], check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit:", COMMIT, "| has_baseline_patch:", os.path.exists(BASELINE_PATCH_DRIVE))

In [ ]:
import os, shutil, subprocess, sys

subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", "main"], check=True)
subprocess.run(["git", "-C", REPO, "reset", "--hard", "origin/main"], check=True)
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()

LOCAL_PATCH_OUT = "/content/lfm3_prose_preserving_sparse.moipatch"
LOCAL_OUT = f"/content/{RUN_NAME}.json"
SCRIPT = f"{REPO}/scripts/experiment_prose_preserving_always_on_lfm3.py"
if not os.path.exists(SCRIPT):
    raise FileNotFoundError(f"Falta {SCRIPT} en {REPO} (commit {COMMIT}).")

cmd = [
    sys.executable, "-u", SCRIPT,
    "--model", BASE,
    "--patch-out", LOCAL_PATCH_OUT,
    "--n-or-bench", str(N_OR),
    "--n-control", str(N_CTRL),
    "--n-traces", str(N_TRACES),
    "--n-prose-traces", str(N_TRACES),
    "--max-new-tokens", str(MAX_NEW_TOK),
    "--batch-size", str(BATCH_SIZE),
    "--out", LOCAL_OUT,
]
if os.path.exists(BASELINE_PATCH_DRIVE):
    cmd.extend(["--baseline-patch", BASELINE_PATCH_DRIVE])

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, cmd)

os.makedirs(os.path.dirname(PATCH_OUT_DRIVE), exist_ok=True)
shutil.copyfile(LOCAL_PATCH_OUT, PATCH_OUT_DRIVE + ".tmp")
os.replace(PATCH_OUT_DRIVE + ".tmp", PATCH_OUT_DRIVE)
print("Parche esparso guardado en Drive:", PATCH_OUT_DRIVE)

In [ ]:
import datetime, json, os, subprocess

res = json.load(open(LOCAL_OUT, encoding="utf-8"))
_pct = lambda v: f"{v:6.1%}" if isinstance(v, (int, float)) else "   n/a"
print("=== COMPARATIVA ALWAYS_ON EN HALF-B (OR-Bench + Redacción Sana) ===")
hdr = f"{'arm':22s} | {'mods':4s} | {'weights':9s} | {'OR recov':8s} | {'healthy_kl':10s} | {'prose_kl':8s} | {'top1_ok':7s} | {'1st16':6s} | {'exact':6s}"
print(hdr)
print("-" * len(hdr))
for name in ("base", "baseline_always_on", "fisher_sparse_any", "fisher_sparse_late", "fisher_ultra_low_kl"):
    m = res["arms"].get(name)
    if not m:
        continue
    print(
        f"{name:22s} | {m['n_edited_modules']:4d} | {m['edited_weights']:9,d} | {_pct(m.get('recovered_of_base_refused')):8s} | "
        f"{m['healthy_kl']:10.5f} | {m['prose_response_kl']:8.5f} | {_pct(m.get('prose_top1_unchanged_rate')):7s} | "
        f"{_pct(m.get('control_first16_exact_match_rate')):6s} | {_pct(m.get('control_exact_match_rate')):6s}"
    )

print("\n--- Módulos elegidos por brazo ---")
for name, meta in res["screening"]["selected"].items():
    print(f"  {name:22s}: {meta['candidate']} | mods={meta['modules']} | cols={meta['n_cols']}")
print("Exported arm:", res["exported_arm"], "| audit fp_ok=", (res.get("audit") or {}).get("fingerprint_ok"))
print("Elapsed total (s):", res["timing_s"]["total"])

res["_final"] = {
    "complete": True,
    "model": MODEL,
    "commit": COMMIT,
    "notebook": "experiment_prose_preserving_always_on_lfm3_colab.ipynb",
    "patch_out_drive": PATCH_OUT_DRIVE,
    "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
    "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
}
for out_dir in (FINALES, RUN_DIR):
    os.makedirs(out_dir, exist_ok=True)
    dst = f"{out_dir}/{RUN_NAME}.json"
    with open(dst + ".tmp", "w", encoding="utf-8") as f:
        json.dump(res, f, indent=2)
    os.replace(dst + ".tmp", dst)
print("\nGuardado final en Drive:", f"{FINALES}/{RUN_NAME}.json")

In [ ]:
AUTO_APAGAR = True
import os, time, json
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst, encoding="utf-8")).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")